In [ ]:
from pathlib import Path
import gc
import re
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.spatial import cKDTree

In [ ]:
base_dir = Path("/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC/Data/ST/Samples")

global_outdir = Path("/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC/Data/ST/Spatial_Niche")
global_outdir.mkdir(parents=True, exist_ok=True)

spatial_key = "spatial_um"
celltype_col = "CellType"
leiden_col = "leiden"
cnv_col = "CNV_subcluster_smooth"

celltype_out_col = "CellType"
niche_out_col = "Spatial-niche"
clone_out_col = "Clone"

unassigned_label = "unassigned"
majority_threshold = 0.5

hex15_side_length = 15.0
hex36_side_length = 36.0
distance_factor = 1.15

max_dist_15 = hex15_side_length * distance_factor
max_dist_36 = hex36_side_length * distance_factor

save_each_sample_full_adata = True
save_combined_obs_only = False


In [ ]:
def get_paths(sample_dir):
    sample_id = sample_dir.name
    return {
        "sample_id": sample_id,
        "adata_15_path": sample_dir / "GLMPCA" / "GLMPCA_adata.h5ad",
        "cell_path": sample_dir / "cell_segment" / "Cell_Segment_addCelltype_adata.h5ad",
        "adata_36_path": sample_dir / "hexagon" / "add_InferCNV_adata_36um.h5ad",
    }


def check_required_fields(adata, obs_cols=None, obsm_keys=None, object_name="adata"):
    obs_cols = [] if obs_cols is None else obs_cols
    obsm_keys = [] if obsm_keys is None else obsm_keys

    missing_obs = [col for col in obs_cols if col not in adata.obs.columns]
    missing_obsm = [key for key in obsm_keys if key not in adata.obsm]

    if missing_obs:
        raise KeyError(f"{object_name} missing obs columns: {missing_obs}")
    if missing_obsm:
        raise KeyError(f"{object_name} missing obsm keys: {missing_obsm}")


def clean_label_series(values, unassigned="unassigned"):
    series = pd.Series(values, copy=True)
    invalid = series.isna()
    series = series.astype(str).str.strip()
    invalid = invalid | series.str.lower().isin(["", "nan", "none", "na", "<na>"])
    series.loc[invalid] = unassigned
    return series.astype(str)


def add_common_metadata(adata, sample_id, original_name_col):
    adata = adata.copy()
    original_names = adata.obs_names.astype(str).to_numpy()
    xy = np.asarray(adata.obsm[spatial_key])[:, :2]

    adata.obs[original_name_col] = original_names
    adata.obs["sample_id"] = sample_id
    adata.obs["x"] = xy[:, 0]
    adata.obs["y"] = xy[:, 1]

    adata.obs_names = [f"{sample_id}__{name}" for name in original_names]
    adata.obs_names_make_unique()
    return adata


def nearest_map_with_threshold(query_xy, ref_xy, ref_ids, ref_values, max_dist, unassigned="unassigned"):
 
    query_xy = np.asarray(query_xy, dtype=float)
    ref_xy = np.asarray(ref_xy, dtype=float)
    ref_ids = np.asarray(ref_ids, dtype=object)
    ref_values = np.asarray(ref_values, dtype=object)

    n_query = query_xy.shape[0]
    mapped_ids = np.full(n_query, unassigned, dtype=object)
    mapped_values = np.full(n_query, unassigned, dtype=object)
    dist = np.full(n_query, np.nan, dtype=float)
    accepted = np.zeros(n_query, dtype=bool)

    if n_query == 0 or ref_xy.shape[0] == 0:
        return mapped_ids, mapped_values, dist, accepted

    tree = cKDTree(ref_xy)
    dist, idx = tree.query(query_xy, k=1)
    accepted = np.isfinite(dist) & (dist <= max_dist)

    mapped_ids[accepted] = ref_ids[idx[accepted]]
    mapped_values[accepted] = ref_values[idx[accepted]]
    return mapped_ids, mapped_values, dist, accepted


def aggregate_majority_label(
    source_obs,
    target_index,
    group_col,
    label_col,
    output_label_col,
    member_prefix,
    threshold=0.5,
    unassigned="unassigned",
):
 
    target_index = pd.Index(target_index.astype(str))
    result = pd.DataFrame(index=target_index)
    result[output_label_col] = unassigned
    result[f"n_{member_prefix}"] = 0
    result[f"n_{member_prefix}_valid_{output_label_col}"] = 0
    result[f"{output_label_col}_fraction"] = 0.0
    result[f"{output_label_col}_tie"] = False

    if source_obs.empty:
        return result

    dat = source_obs[[group_col, label_col]].copy()
    dat[group_col] = clean_label_series(dat[group_col], unassigned=unassigned).to_numpy()
    dat[label_col] = clean_label_series(dat[label_col], unassigned=unassigned).to_numpy()

    dat = dat[dat[group_col] != unassigned].copy()
    if dat.empty:
        return result

    total_members = dat.groupby(group_col, observed=True).size()
    result.loc[result.index.intersection(total_members.index), f"n_{member_prefix}"] = (
        total_members.reindex(result.index.intersection(total_members.index)).astype(int).to_numpy()
    )

    valid = dat[dat[label_col] != unassigned].copy()
    if valid.empty:
        return result

    label_counts = (
        valid.groupby([group_col, label_col], observed=True)
        .size()
        .rename("n_label")
        .reset_index()
    )

    valid_members = valid.groupby(group_col, observed=True).size().rename("n_valid")
    max_count = label_counts.groupby(group_col, observed=True)["n_label"].transform("max")
    winners = label_counts[label_counts["n_label"] == max_count].copy()

    winner_summary = (
        winners.groupby(group_col, observed=True)
        .agg(
            winner_label=(label_col, "first"),
            max_count=("n_label", "first"),
            n_winners=(label_col, "size"),
        )
    )
    winner_summary = winner_summary.join(valid_members, how="left")
    winner_summary["fraction"] = winner_summary["max_count"] / winner_summary["n_valid"]
    winner_summary["assigned"] = (
        (winner_summary["fraction"] >= threshold) &
        (winner_summary["n_winners"] == 1)
    )

    common = result.index.intersection(winner_summary.index)
    ws = winner_summary.reindex(common)

    result.loc[common, f"n_{member_prefix}_valid_{output_label_col}"] = ws["n_valid"].astype(int).to_numpy()
    result.loc[common, f"{output_label_col}_fraction"] = ws["fraction"].astype(float).to_numpy()
    result.loc[common, f"{output_label_col}_tie"] = (ws["n_winners"] > 1).to_numpy()

    assigned_index = ws.index[ws["assigned"]]
    result.loc[assigned_index, output_label_col] = ws.loc[assigned_index, "winner_label"].astype(str).to_numpy()
    return result


def make_obs_only_adata(adata, spatial_key="spatial"):
    obs_only = ad.AnnData(obs=adata.obs.copy())
    if spatial_key in adata.obsm:
        obs_only.obsm[spatial_key] = np.asarray(adata.obsm[spatial_key])[:, :2].copy()
    return obs_only


def save_adata(adata, out_path, save_full=True):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if save_full:
        adata.write_h5ad(out_path)
    else:
        make_obs_only_adata(adata, spatial_key=spatial_key).write_h5ad(out_path)


def save_obs_table(adata, out_path, index_name="obs_name"):
    table = adata.obs.copy().reset_index().rename(columns={"index": index_name})
    table.to_csv(out_path, sep="\t", index=False)


def safe_distance_summary(dist):
    dist = np.asarray(dist, dtype=float)
    dist = dist[np.isfinite(dist)]
    if dist.size == 0:
        return np.nan, np.nan
    return float(np.median(dist)), float(np.percentile(dist, 95))
 

In [ ]:

def process_one_sample(sample_dir):
    paths = get_paths(sample_dir)
    sample_id = paths["sample_id"]
 
    print(f"Processing {sample_id}")
 
    adata_15 = sc.read_h5ad(paths["adata_15_path"])
    cell_adata = sc.read_h5ad(paths["cell_path"])

    mu_scale = cell_adata.uns['mu_scale']['spatial_manual_fine']
    adata_15.obsm['spatial_um'] = adata_15.obsm['spatial'] / mu_scale

    check_required_fields(cell_adata, obs_cols=[celltype_col], obsm_keys=[spatial_key], object_name="cell_adata")
    check_required_fields(adata_15, obs_cols=[leiden_col], obsm_keys=[spatial_key], object_name="adata_15")
 
    original_celltype = clean_label_series(cell_adata.obs[celltype_col], unassigned=unassigned_label)
    keep_cell = original_celltype.str.lower().to_numpy() != "conflict"
    cell_adata = cell_adata[keep_cell].copy()
    cell_adata.obs[celltype_out_col] = clean_label_series(
        cell_adata.obs[celltype_col], unassigned=unassigned_label
    ).to_numpy()

    has_36um_cnv = paths["adata_36_path"].exists()
    if has_36um_cnv:
        adata_36 = sc.read_h5ad(paths["adata_36_path"])
        check_required_fields(adata_36, obs_cols=[cnv_col], obsm_keys=[spatial_key], object_name="adata_36")
    else:
        adata_36 = None
 
    cell_adata = add_common_metadata(cell_adata, sample_id, "original_cell_obs_name")

    if "cell_id" in cell_adata.obs.columns:
        cell_adata.obs["cell_id_for_export"] = cell_adata.obs["cell_id"].astype(str).to_numpy()
    elif "cell_ID_mask" in cell_adata.obs.columns:
        cell_adata.obs["cell_id_for_export"] = cell_adata.obs["cell_ID_mask"].astype(str).to_numpy()
    else:
        cell_adata.obs["cell_id_for_export"] = cell_adata.obs["original_cell_obs_name"].astype(str).to_numpy()

    adata_15 = add_common_metadata(adata_15, sample_id, "original_hex15_obs_name")

    if has_36um_cnv:
        adata_36 = add_common_metadata(adata_36, sample_id, "original_hex36_obs_name")

 
    cell_adata.obs[celltype_out_col] = clean_label_series(cell_adata.obs[celltype_out_col], unassigned=unassigned_label).to_numpy()
    adata_15.obs[niche_out_col] = clean_label_series(adata_15.obs[leiden_col], unassigned=unassigned_label).to_numpy()

    if has_36um_cnv:
        adata_36.obs[clone_out_col] = clean_label_series(adata_36.obs[cnv_col], unassigned=unassigned_label).to_numpy()

    cell_xy = np.asarray(cell_adata.obsm[spatial_key])[:, :2]
    hex15_xy = np.asarray(adata_15.obsm[spatial_key])[:, :2]

    if has_36um_cnv:
        hex36_xy = np.asarray(adata_36.obsm[spatial_key])[:, :2]

    # 2.3 cell -> 15um
    matched_hex15_id, matched_niche, dist15, accepted15 = nearest_map_with_threshold(
        query_xy=cell_xy,
        ref_xy=hex15_xy,
        ref_ids=adata_15.obs_names.astype(str),
        ref_values=adata_15.obs[niche_out_col].astype(str).to_numpy(),
        max_dist=max_dist_15,
        unassigned=unassigned_label,
    )

    # 2.4 cell -> 36um
    if has_36um_cnv:
        matched_hex36_id, matched_clone, dist36, accepted36 = nearest_map_with_threshold(
            query_xy=cell_xy,
            ref_xy=hex36_xy,
            ref_ids=adata_36.obs_names.astype(str),
            ref_values=adata_36.obs[clone_out_col].astype(str).to_numpy(),
            max_dist=max_dist_36,
            unassigned=unassigned_label,
        )
    else:
        n_cell = cell_adata.n_obs
        matched_hex36_id = np.full(n_cell, unassigned_label, dtype=object)
        matched_clone = np.full(n_cell, unassigned_label, dtype=object)
        dist36 = np.full(n_cell, np.nan, dtype=float)
        accepted36 = np.zeros(n_cell, dtype=bool)

    # merged cell-level：each cell have CellType、Spatial-niche、Clone
    merged_cell_adata = cell_adata.copy()
    merged_cell_adata.obs["matched_hex15_id"] = matched_hex15_id.astype(str)
    merged_cell_adata.obs[niche_out_col] = clean_label_series(matched_niche, unassigned=unassigned_label).to_numpy()
    merged_cell_adata.obs["dist_to_hex15"] = dist15
    merged_cell_adata.obs["hex15_match_accepted"] = accepted15

    merged_cell_adata.obs["matched_hex36_id"] = matched_hex36_id.astype(str)
    merged_cell_adata.obs[clone_out_col] = clean_label_series(matched_clone, unassigned=unassigned_label).to_numpy()
    merged_cell_adata.obs["dist_to_hex36"] = dist36
    merged_cell_adata.obs["hex36_match_accepted"] = accepted36
 
    agg_celltype_15 = aggregate_majority_label(
        source_obs=merged_cell_adata.obs,
        target_index=adata_15.obs_names,
        group_col="matched_hex15_id",
        label_col=celltype_out_col,
        output_label_col=celltype_out_col,
        member_prefix="cells",
        threshold=majority_threshold,
        unassigned=unassigned_label,
    )

    for col in agg_celltype_15.columns:
        adata_15.obs[col] = agg_celltype_15[col].reindex(adata_15.obs_names).to_numpy()

    if has_36um_cnv:
        hex15_matched_hex36_id, hex15_matched_clone, dist15_to_36, accepted15_to_36 = nearest_map_with_threshold(
            query_xy=hex15_xy,
            ref_xy=hex36_xy,
            ref_ids=adata_36.obs_names.astype(str),
            ref_values=adata_36.obs[clone_out_col].astype(str).to_numpy(),
            max_dist=max_dist_36,
            unassigned=unassigned_label,
        )
    else:
        n_hex15 = adata_15.n_obs
        hex15_matched_hex36_id = np.full(n_hex15, unassigned_label, dtype=object)
        hex15_matched_clone = np.full(n_hex15, unassigned_label, dtype=object)
        dist15_to_36 = np.full(n_hex15, np.nan, dtype=float)
        accepted15_to_36 = np.zeros(n_hex15, dtype=bool)

    adata_15.obs["matched_hex36_id"] = hex15_matched_hex36_id.astype(str)
    adata_15.obs[clone_out_col] = clean_label_series(
        hex15_matched_clone, unassigned=unassigned_label
    ).to_numpy()
    adata_15.obs["dist_to_hex36"] = dist15_to_36
    adata_15.obs["hex36_match_accepted"] = accepted15_to_36
 
    if has_36um_cnv:
        agg_celltype_36 = aggregate_majority_label(
            source_obs=merged_cell_adata.obs,
            target_index=adata_36.obs_names,
            group_col="matched_hex36_id",
            label_col=celltype_out_col,
            output_label_col=celltype_out_col,
            member_prefix="cells",
            threshold=majority_threshold,
            unassigned=unassigned_label,
        )

        agg_niche_36 = aggregate_majority_label(
            source_obs=adata_15.obs,
            target_index=adata_36.obs_names,
            group_col="matched_hex36_id",
            label_col=niche_out_col,
            output_label_col=niche_out_col,
            member_prefix="hex15",
            threshold=majority_threshold,
            unassigned=unassigned_label,
        )

        for col in agg_celltype_36.columns:
            adata_36.obs[col] = agg_celltype_36[col].reindex(adata_36.obs_names).to_numpy()

        for col in agg_niche_36.columns:
            adata_36.obs[col] = agg_niche_36[col].reindex(adata_36.obs_names).to_numpy()

 
    dist15_median, dist15_p95 = safe_distance_summary(dist15)
    dist36_median, dist36_p95 = safe_distance_summary(dist36)
    dist15_to_36_median, dist15_to_36_p95 = safe_distance_summary(dist15_to_36)

    summary = {
        "sample_id": sample_id,
        "has_36um_cnv": has_36um_cnv,
        "n_cells": cell_adata.n_obs,
        "n_15um_hex": adata_15.n_obs,
        "n_36um_hex": adata_36.n_obs if has_36um_cnv else 0,
        "max_dist_15": max_dist_15,
        "max_dist_36": max_dist_36,
        "cell_to_hex15_n_accepted": int(accepted15.sum()),
        "cell_to_hex15_pct_accepted": float(accepted15.mean() * 100) if accepted15.size else np.nan,
        "cell_to_hex15_dist_median": dist15_median,
        "cell_to_hex15_dist_p95": dist15_p95,
        "cell_to_hex36_n_accepted": int(accepted36.sum()),
        "cell_to_hex36_pct_accepted": float(accepted36.mean() * 100) if accepted36.size else np.nan,
        "cell_to_hex36_dist_median": dist36_median,
        "cell_to_hex36_dist_p95": dist36_p95,
        "hex15_to_hex36_n_accepted": int(accepted15_to_36.sum()),
        "hex15_to_hex36_pct_accepted": float(accepted15_to_36.mean() * 100) if accepted15_to_36.size else np.nan,
        "hex15_to_hex36_dist_median": dist15_to_36_median,
        "hex15_to_hex36_dist_p95": dist15_to_36_p95,
        "hex15_CellType_unassigned_pct": float((adata_15.obs[celltype_out_col] == unassigned_label).mean() * 100),
        "hex15_Clone_unassigned_pct": float((adata_15.obs[clone_out_col] == unassigned_label).mean() * 100),
        "hex36_CellType_unassigned_pct": (
            float((adata_36.obs[celltype_out_col] == unassigned_label).mean() * 100)
            if has_36um_cnv else np.nan
        ),
        "hex36_Spatial_niche_unassigned_pct": (
            float((adata_36.obs[niche_out_col] == unassigned_label).mean() * 100)
            if has_36um_cnv else np.nan
        ),
    }

    print(f"cell -> 15um accepted: {accepted15.sum()} / {accepted15.size} ({summary['cell_to_hex15_pct_accepted']:.2f}%)")
    if has_36um_cnv:
        print(f"cell -> 36um accepted: {accepted36.sum()} / {accepted36.size} ({summary['cell_to_hex36_pct_accepted']:.2f}%)")
        print(f"15um -> 36um accepted: {accepted15_to_36.sum()} / {accepted15_to_36.size} ({summary['hex15_to_hex36_pct_accepted']:.2f}%)")
    else:
        print("36um file missing: Clone-related labels are set to unassigned")

    return {
        "cell_adata": merged_cell_adata,
        "niche_adata": adata_15,
        "clone_adata": adata_36, 
        "summary": summary,
    }



In [ ]:
sample_dirs = sorted([
    p for p in base_dir.iterdir()
    if p.is_dir()
    and not p.name.startswith(".") 
])

all_cell_level = []
all_niche_level = []
all_clone_level = []

summary_list = []
failed_list = []

for sample_dir in sample_dirs:
    sample_id = sample_dir.name
    paths = get_paths(sample_dir)

    required_files = [paths["adata_15_path"], paths["cell_path"],]

    missing = [str(path)
               for path in required_files
               if not path.exists()
               ]

    if missing:
        print(f"\nSkip {sample_id}, missing required files:")

        for path in missing:
            print(" ", path)

        failed_list.append({"sample_id": sample_id, "status": "missing_files", "missing": ";".join(missing),})
        continue

    try:
        result = process_one_sample(sample_dir)
 
        # CellType、Spatial-niche、Clone
        cell_adata = result["cell_adata"]
        niche_adata = result["niche_adata"]
        clone_adata = result["clone_adata"]
        summary = result["summary"]

        sample_outdir = sample_dir / "integrated"
        sample_outdir.mkdir(parents=True, exist_ok=True,)

        out_cell = sample_outdir / "cell_adata.h5ad"
        save_adata(cell_adata, out_cell, save_full=save_each_sample_full_adata,)

        out_niche = sample_outdir / "niche_adata.h5ad"
        save_adata(niche_adata, out_niche, save_full=save_each_sample_full_adata,)

        if clone_adata is not None:
            out_clone = sample_outdir / "clone_adata.h5ad"
            save_adata(clone_adata, out_clone, save_full=save_each_sample_full_adata,)
        else:
            out_clone = None


        if out_clone is not None:
            print(" ", out_clone)
        else:
            print("  clone_adata.h5ad not saved: 36 μm object missing")
            
        if save_combined_obs_only:
            all_cell_level.append(make_obs_only_adata(cell_adata, spatial_key=spatial_key,))
            all_niche_level.append(make_obs_only_adata(niche_adata,spatial_key=spatial_key,))
            if clone_adata is not None:
                all_clone_level.append(make_obs_only_adata(clone_adata,spatial_key=spatial_key,))
        else:
            all_cell_level.append(cell_adata.copy())
            all_niche_level.append(niche_adata.copy())
            if clone_adata is not None:
                all_clone_level.append(clone_adata.copy())

        summary_list.append(summary)
 
        gc.collect()

    except Exception as e:
        print(f"\nFailed {sample_id}: {repr(e)}")
        failed_list.append({"sample_id": sample_id, "status": "failed", "error": repr(e),})
        continue

print("\nFinished.")
print("Success:", len(summary_list))
print("Failed:", len(failed_list))

In [ ]:
def concat_and_save(adatas, out_path):
    combined = ad.concat(adatas, axis=0, join="outer", merge="first", uns_merge=None, index_unique=None,)
    combined.obs_names_make_unique()
    combined.write_h5ad(out_path)
    print("Saved:", out_path)
    print(combined)

    return combined


In [ ]:
summary_df = pd.DataFrame(summary_list)
summary_path = global_outdir / "all_samples_multiresolution_summary.tsv"
summary_df.to_csv(summary_path, sep="\t", index=False,)
 

In [ ]:
combined_cell = ad.concat(all_cell_level, axis=0, join="outer", merge="first", uns_merge=None, index_unique=None,)
combined_niche = ad.concat(all_niche_level, axis=0, join="outer", merge="first", uns_merge=None, index_unique=None,)
combined_clone = ad.concat(all_clone_level, axis=0, join="outer", merge="first", uns_merge=None, index_unique=None,)

In [ ]:
for name, adata_obj in {
    "cell": combined_cell,
    "niche": combined_niche,
    "clone": combined_clone,
}.items():

    if adata_obj is None:
        continue

    if "mt" in adata_obj.var.columns:
        gene_names = adata_obj.var_names.astype(str)

        adata_obj.var["mt"] = (
            gene_names.str.upper().str.startswith("MT-")
        ).astype(bool)

        print(
            name,
            adata_obj.var["mt"].dtype,
            adata_obj.var["mt"].value_counts(dropna=False).to_dict(),
        )

In [ ]:
pat = re.compile(r'^(ENSG|ENST|ENSP|ENS)\d+(\.\d+)?$') 
mask = ~combined_cell.var_names.astype(str).str.match(pat)
combined_cell = combined_cell[:, mask].copy()
mask = ~combined_niche.var_names.astype(str).str.match(pat)
combined_niche = combined_niche[:, mask].copy()
mask = ~combined_clone.var_names.astype(str).str.match(pat)
combined_clone = combined_clone[:, mask].copy()

In [ ]:
combined_cell.write_h5ad(global_outdir / "all_samples_cell_adata.h5ad")
combined_niche.write_h5ad(global_outdir / "all_samples_niche_adata.h5ad")
combined_clone.write_h5ad(global_outdir / "all_samples_clone_adata.h5ad")

In [ ]:
combined_niche = sc.read_h5ad("/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC/Data/ST/Spatial_Niche/all_samples_niche_adata.h5ad")

In [ ]:
import scipy.sparse as sp
import os
from scipy.io import mmwrite
import gzip

X = combined_niche.layers["raw"]
var_names = combined_niche.var_names

if not sp.issparse(X):
    X = sp.csr_matrix(np.asarray(X))
X = X.tocsr().astype(np.float32)
if X.nnz > 0:
    X.data = np.clip(X.data, 0, None)

outdir = '/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC/Data/ST/Spatial_Niche/raw_matrix/'

mtx_gz = os.path.join(outdir, "counts.mtx.gz")
with gzip.open(mtx_gz, "wb", compresslevel=1) as f:
    mmwrite(f, X)    

pd.Series(var_names, name="gene").to_csv(os.path.join(outdir, "genes.tsv"), sep="\t", index=False, header=False)

combined_niche.obs["barcode"] = combined_niche.obs.index
barcodes = combined_niche.obs["barcode"].astype(str).values   
pd.Series(barcodes).to_csv(
    os.path.join(outdir, "barcodes.tsv"),
    sep="\t", index=False, header=False
)
combined_niche.obs.to_csv(os.path.join(outdir, "metadata.csv"))
print(f"✅ Finish {outdir}")
